# Telco Customer Churn Analysis

This notebook reproduces the main analysis from `telco_churn_analysis.py`: data cleaning, model comparison, customer segmentation, revenue risk, and governance clause retrieval.

In [ ]:
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_URL = "https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
PROTECTED_PREFIXES = ("gender", "seniorcitizen", "partner", "dependents")

## 1. Load and clean the data

In [ ]:
response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()
data = pd.read_csv(StringIO(response.text))
data.columns = data.columns.str.strip()
text_columns = data.select_dtypes(include=["object"]).columns
data[text_columns] = data[text_columns].apply(lambda column: column.str.strip())
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
zero_tenure = data["TotalCharges"].isna() & data["tenure"].eq(0)
data.loc[zero_tenure, "TotalCharges"] = 0
data["TotalCharges"] = data["TotalCharges"].fillna(data["TotalCharges"].median())
print(f"Rows: {len(data):,}")
print(f"Overall churn rate: {(data['Churn'] == 'Yes').mean():.2%}")
data.head()

## 2. Compare churn models

In [ ]:
y = (data["Churn"] == "Yes").astype(int)
X = data.drop(columns=["Churn", "customerID"])
X = pd.get_dummies(X, drop_first=True).astype(float)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_all_scaled = scaler.transform(X)

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
forest_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, class_weight="balanced"
)
logistic_model.fit(X_train_scaled, y_train)
forest_model.fit(X_train, y_train)

metrics = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "train_accuracy": accuracy_score(y_train, logistic_model.predict(X_train_scaled)),
        "test_accuracy": accuracy_score(y_test, logistic_model.predict(X_test_scaled)),
        "test_roc_auc": roc_auc_score(y_test, logistic_model.predict_proba(X_test_scaled)[:, 1]),
    },
    {
        "model": "Random Forest",
        "train_accuracy": accuracy_score(y_train, forest_model.predict(X_train)),
        "test_accuracy": accuracy_score(y_test, forest_model.predict(X_test)),
        "test_roc_auc": roc_auc_score(y_test, forest_model.predict_proba(X_test)[:, 1]),
    },
])
display(metrics.round(4))
print(classification_report(y_test, logistic_model.predict(X_test_scaled)))
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

## 3. Segment customers and estimate revenue at risk

In [ ]:
data["predicted_churn_probability"] = logistic_model.predict_proba(X_all_scaled)[:, 1]
segment_inputs = data[["tenure", "MonthlyCharges", "predicted_churn_probability"]]
data["segment"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(
    StandardScaler().fit_transform(segment_inputs)
)
segment_summary = data.groupby("segment").agg(
    customers=("segment", "size"),
    avg_tenure=("tenure", "mean"),
    avg_monthly_charges=("MonthlyCharges", "mean"),
    avg_churn_probability=("predicted_churn_probability", "mean"),
)
segment_summary["estimated_monthly_revenue_at_risk"] = (
    segment_summary["customers"]
    * segment_summary["avg_monthly_charges"]
    * segment_summary["avg_churn_probability"]
)
tenure_midpoint = segment_summary["avg_tenure"].median()
spend_midpoint = segment_summary["avg_monthly_charges"].median()
risk_midpoint = segment_summary["avg_churn_probability"].median()

def segment_name(row):
    tenure_name = "New" if row["avg_tenure"] < tenure_midpoint else "Mature"
    spend_name = "Low-Spend" if row["avg_monthly_charges"] < spend_midpoint else "High-Spend"
    risk_name = "High-Risk" if row["avg_churn_probability"] >= risk_midpoint else "Low-Risk"
    return f"{tenure_name}, {spend_name}, {risk_name}"

segment_summary["segment_name"] = segment_summary.apply(segment_name, axis=1)
segment_summary = segment_summary.sort_values("estimated_monthly_revenue_at_risk", ascending=False)
display(segment_summary.round(2))
segment_summary.to_csv(OUTPUT_DIR / "segment_summary.csv")

## 4. Retrieve grounded retention clauses

In [ ]:
CLAUSES = {
    1: "High risk (probability >= 0.70): offer a loyalty discount and callback within 48 hours.",
    2: "Moderate risk (0.40-0.70): send a targeted email with an underused service or upgrade offer.",
    3: "New customer (tenure < 3 months): route to onboarding instead of standard retention.",
    4: "Non-discrimination: never state or imply that protected demographic attributes contributed to risk.",
}

def retrieve_clauses(probability, tenure):
    retrieved = []
    if probability >= 0.70:
        retrieved.append(CLAUSES[1])
    elif probability >= 0.40:
        retrieved.append(CLAUSES[2])
    if tenure < 3:
        retrieved.append(CLAUSES[3])
    retrieved.append(CLAUSES[4])
    return retrieved

def is_protected_feature(feature_name):
    return feature_name.lower().startswith(PROTECTED_PREFIXES)

customer_index = data["predicted_churn_probability"].idxmax()
customer_probability = float(data.loc[customer_index, "predicted_churn_probability"])
customer_tenure = float(data.loc[customer_index, "tenure"])
contributions = scaler.transform(X.loc[[customer_index]])[0] * logistic_model.coef_[0]
contribution_series = pd.Series(contributions, index=X.columns)
top_features = contribution_series.abs().sort_values(ascending=False).head(3).index
safe_top_features = [feature for feature in top_features if not is_protected_feature(feature)]
retrieved = retrieve_clauses(customer_probability, customer_tenure)
advisory_inputs = {
    "customer_index": int(customer_index),
    "risk_probability": customer_probability,
    "tenure": customer_tenure,
    "safe_top_features": list(safe_top_features),
    "retrieved_clause_numbers": [number for number, clause in CLAUSES.items() if clause in retrieved],
}
print(advisory_inputs)
pd.DataFrame([advisory_inputs]).to_json(OUTPUT_DIR / "advisory_inputs.json", orient="records", indent=2)
(OUTPUT_DIR / "retrieved_clauses.txt").write_text("\n\n".join(retrieved), encoding="utf-8")